In [13]:
import numpy as np
import pandas as pd
import scipy as sp

import jax
import jax.numpy as jnp

import altair as alt

In [14]:
x = np.linspace(0, 100, 100)
log_exposure = np.exp(-x / 10)
f = 0.5 * np.sin(x / 20) + 0.5 * np.cos(x / 10)
rate = np.exp(log_exposure + f) 

x_coarse = np.repeat(np.arange(2.5, 98.5, 5), 5)
log_exposure_coarse = np.exp(-x_coarse / 10)
f_coarse = 0.5 * np.sin(x_coarse / 20) + 0.5 * np.cos(x_coarse / 10)
rate_coarse = np.exp(log_exposure_coarse + f_coarse)

y = np.random.poisson(rate_coarse, size=(100,))

df_data = pd.DataFrame({
	'x': x,
	'log_exposure': log_exposure,
    'log_exposure_coarse': log_exposure_coarse,
	'f': f,
	'rate': rate,
    'rate_coarse': rate_coarse,
	'y': y
})


In [15]:
rate = (
  alt.Chart(df_data)
  .mark_line(color='black', strokeDash=[5, 5])
  .encode(
		x=alt.X('x', title='x'),
		y=alt.Y('rate', title='Rate'),
	)
)

rate_coarse = (
  alt.Chart(df_data)
  .mark_line(color='red')
  .encode(
		x=alt.X('x', title='x'),
		y=alt.Y('rate_coarse', title='Coarse Rate'),
	)
)

count = alt.Chart(df_data).mark_point().encode(
	x=alt.X('x', title='x'),
	y=alt.Y('y', title='Count'),
)

(rate + count + rate_coarse).properties(width=600)

alt.LayerChart(...)

In [16]:
df_data['range'] = pd.cut(df_data['x'], bins=15, right=False)

tmp = df_data.groupby('range', observed=True).agg(
	y_agg = ('y', 'sum'),
	N = ('y', 'count')
).reset_index()

df_data = df_data.merge(tmp, on='range', how='left')
df_data['rate_agg'] = df_data['y_agg'] / df_data['N']
df_data.head(5)

,x,log_exposure,log_exposure_coarse,f,rate,rate_coarse,y,range,y_agg,N,rate_agg
0,0.000000,1.000000,0.778801,0.500000,4.481689,3.764422,3,"[0.0, 6.667)",22,7,3.142857
1,1.010101,0.903924,0.778801,0.522693,4.164587,3.764422,4,"[0.0, 6.667)",22,7,3.142857
2,2.020202,0.817078,0.778801,0.540251,3.885801,3.764422,6,"[0.0, 6.667)",22,7,3.142857
3,3.030303,0.738577,0.778801,0.552686,3.637378,3.764422,3,"[0.0, 6.667)",22,7,3.142857
4,4.040404,0.667617,0.778801,0.560064,3.413307,3.764422,3,"[0.0, 6.667)",22,7,3.142857


In [29]:
# Create a clean dataset without the Interval objects for plotting
df_plot = df_data.drop(columns=['range'])

rate_coarse = (
  alt.Chart(df_plot)
  .mark_line(color='red')
  .encode(
		x=alt.X('x', title='x'),
		y=alt.Y('rate_coarse', title='Coarse Rate'),
	)
)

count = alt.Chart(df_plot).mark_point().encode(
	x=alt.X('x', title='x'),
	y=alt.Y('y', title='Count'),
)

rate_agg = (
  alt.Chart(df_plot)
  .mark_line(color='blue', strokeDash=[5, 5])  # Changed color and added dashed line to distinguish
  .encode(
		x=alt.X('x', title='x'),
		y=alt.Y('rate_agg', title='Aggregated Rate'),
	)
)

(rate_agg + count + rate_coarse).properties(width=600)

alt.LayerChart(...)

In [18]:
import numpyro
from numpyro.contrib.hsgp.approximation import hsgp_squared_exponential
from numpyro.contrib.hsgp.laplacian import eigenfunctions
from numpyro.contrib.hsgp.spectral_densities import (
	diag_spectral_density_squared_exponential
)
import numpyro.distributions as dist
from numpyro.infer import SVI, MCMC, NUTS, Predictive

In [19]:
def create_interval_index_array(intervals_series, grid_start=0, grid_end=None, grid_step=1):
    """
    Create an index array that maps intervals to points on a fine integer grid.
    
    Parameters:
    -----------
    intervals_series : pandas.Series
        A pandas Series of categorical dtype containing pd.Interval objects
        that are closed on the left and open on the right [left, right).
    grid_start : int, default 0
        The starting point of the integer grid.
    grid_end : int, optional
        The ending point of the integer grid. If None, inferred from intervals.
    grid_step : int, default 1
        The step size for the integer grid.
    
    Returns:
    --------
    numpy.ndarray
        An index array where each element corresponds to a grid point and
        contains the index of the interval that contains that grid point.
        Points not in any interval get index -1.
    
    Example:
    --------
    >>> import pandas as pd
    >>> import numpy as np
    >>> 
    >>> # Create some intervals
    >>> x = np.linspace(0, 10, 11)
    >>> intervals = pd.cut(x, bins=3, right=False)
    >>> 
    >>> # Create index array for fine grid from 0 to 10
    >>> index_array = create_interval_index_array(intervals, 0, 10)
    >>> print(index_array)
    """
    
    # Get unique intervals and their categorical codes
    unique_intervals = intervals_series.cat.categories
    interval_codes = intervals_series.cat.codes
    
    # Determine grid bounds if not provided
    if grid_end is None:
        max_right = max(interval.right for interval in unique_intervals)
        grid_end = int(np.floor(max_right))

    # Create the fine integer grid
    grid_points = np.arange(grid_start, grid_end + grid_step, grid_step)
    
    # Initialize index array with -1 (indicating no interval contains the point)
    index_array = np.full(len(grid_points), -1, dtype=int)
    
    # For each unique interval, find which grid points it contains
    for i, interval in enumerate(unique_intervals):
        # Find grid points that fall within this interval [left, right)
        mask = (grid_points >= interval.left) & (grid_points < interval.right)
        
        # Set the index array to the categorical code for this interval
        index_array[mask] = i
    
    return index_array


def create_mapping_matrix(intervals_series, grid_start=0, grid_end=None, grid_step=1):
    """
    Create a mapping matrix that aggregates values from a fine grid to intervals.
    
    Parameters:
    -----------
    intervals_series : pandas.Series
        A pandas Series of categorical dtype containing pd.Interval objects.
    grid_start : int, default 0
        The starting point of the integer grid.
    grid_end : int, optional
        The ending point of the integer grid. If None, inferred from intervals.
    grid_step : int, default 1
        The step size for the integer grid.
    
    Returns:
    --------
    numpy.ndarray
        A matrix of shape (n_intervals, n_grid_points) where each row corresponds
        to an interval and contains 1s for grid points within that interval.
        This can be used for aggregation: aggregated_values = matrix @ grid_values
    """
    
    # Get the index array
    index_array = create_interval_index_array(intervals_series, grid_start, grid_end, grid_step)
    
    # Get number of unique intervals
    n_intervals = len(intervals_series.cat.categories)
    n_grid_points = len(index_array)
    
    # Create mapping matrix
    mapping_matrix = np.zeros((n_intervals, n_grid_points), dtype=int)
    
    # Fill the mapping matrix
    for grid_idx, interval_idx in enumerate(index_array):
        if interval_idx >= 0:  # Valid interval index
            mapping_matrix[interval_idx, grid_idx] = 1
    
    return mapping_matrix

In [20]:
# Standardize x
x_coarse_std = (x_coarse - np.mean(x_coarse)) / np.std(x_coarse)

mapping_matrix = create_mapping_matrix(
  df_data['range'],
  grid_start=0,
  grid_end=99,
  grid_step=1
)

In [21]:
def model(x, log_exposure, map_matrix, L, M, non_centered, y=None):
  # --- Priors ---
  beta = numpyro.sample('baseline', dist.Normal(0, 10))
  sigma = numpyro.sample('sigma', dist.InverseGamma(5, 5))
  lenscale = numpyro.sample('lenscale', dist.InverseGamma(5, 5))
  
  # --- Parameterization ---
  f = hsgp_squared_exponential(x=x, alpha=sigma, length=lenscale, ell=L, m=M, non_centered=non_centered)
  
  # --- Likelihood ---
  log_rate = log_exposure + (beta + f)
  rate = numpyro.deterministic('rate', jnp.exp(log_rate))
  mu_agg = (map_matrix @ rate)
  numpyro.sample('y', dist.Poisson(mu_agg), obs=y)

In [22]:
df_train = df_data.groupby('range').agg(
	y_agg=('y', 'sum'),
	N=('y', 'count')
).reset_index()

y_agg = df_train['y_agg'].values
N = df_train['N'].values

/var/folders/r0/qp6_p2111v1dzb0fg2bbfqsr0000gn/T/ipykernel_15338/2230603012.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_train = df_data.groupby('range').agg(


In [23]:
sampler = NUTS(model)
mcmc = MCMC(sampler, num_warmup=500, num_samples=1_000, num_chains=4)

rng_key, rng_subkey = jax.random.split(jax.random.PRNGKey(42))

L = 2.0
M = 30
non_centered = True

mcmc.run(rng_subkey, 
	x=x_coarse_std, 
	log_exposure=df_data['log_exposure_coarse'].values, 
	map_matrix=mapping_matrix, 
	L=L, 
	M=M, 
	non_centered=non_centered, 
	y=y_agg
)

/var/folders/r0/qp6_p2111v1dzb0fg2bbfqsr0000gn/T/ipykernel_15338/3725013647.py:2: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(sampler, num_warmup=500, num_samples=1_000, num_chains=4)
sample: 100%|██████████| 1500/1500 [00:02<00:00, 504.76it/s, 23 steps of size 8.44e-02. acc. prob=0.89] 


In [24]:
import arviz as az
idata = az.from_numpyro(mcmc)

In [25]:
df_sum = az.summary(
	data=idata,
	var_names=['rate']
)
df_sum.head(10)

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
rate[0],3.226,0.617,2.082,4.353,0.019,0.012,1132.0,1590.0,1.0
rate[1],3.226,0.617,2.082,4.353,0.019,0.012,1132.0,1590.0,1.0
rate[2],3.226,0.617,2.082,4.353,0.019,0.012,1132.0,1590.0,1.0
rate[3],3.226,0.617,2.082,4.353,0.019,0.012,1132.0,1590.0,1.0
rate[4],3.226,0.617,2.082,4.353,0.019,0.012,1132.0,1590.0,1.0
rate[5],2.481,0.415,1.707,3.227,0.013,0.008,1027.0,1598.0,1.0
rate[6],2.481,0.415,1.707,3.227,0.013,0.008,1027.0,1598.0,1.0
rate[7],2.481,0.415,1.707,3.227,0.013,0.008,1027.0,1598.0,1.0
rate[8],2.481,0.415,1.707,3.227,0.013,0.008,1027.0,1598.0,1.0
rate[9],2.481,0.415,1.707,3.227,0.013,0.008,1027.0,1598.0,1.0


In [26]:
df_plot['rate_est'] = df_sum['mean'].values

In [27]:
df_plot.head(5)

,x,log_exposure,log_exposure_coarse,f,rate,rate_coarse,y,y_agg,N,rate_agg,rate_est
0,0.000000,1.000000,0.778801,0.500000,4.481689,3.764422,3,22,7,3.142857,3.226
1,1.010101,0.903924,0.778801,0.522693,4.164587,3.764422,4,22,7,3.142857,3.226
2,2.020202,0.817078,0.778801,0.540251,3.885801,3.764422,6,22,7,3.142857,3.226
3,3.030303,0.738577,0.778801,0.552686,3.637378,3.764422,3,22,7,3.142857,3.226
4,4.040404,0.667617,0.778801,0.560064,3.413307,3.764422,3,22,7,3.142857,3.226


In [28]:
rate_coarse = (
  alt.Chart(df_plot)
  .mark_line(color='red')
  .encode(
		x=alt.X('x', title='x'),
		y=alt.Y('rate_coarse', title='Coarse Rate'),
	)
)

count = alt.Chart(df_plot).mark_point().encode(
	x=alt.X('x', title='x'),
	y=alt.Y('y', title='Count'),
)

rate_est = (
  alt.Chart(df_plot)
	.mark_line(color='green')
	.encode(
		x=alt.X('x', title='x'),
		y=alt.Y('rate_est', title='Estimated Rate'),
	)
)

rate_agg = (
  alt.Chart(df_plot)
  .mark_line(color='blue', strokeDash=[5, 5])
  .encode(
		x=alt.X('x', title='x'),
		y=alt.Y('rate_agg', title='Aggregated Rate'),
	)
)

# (rate_agg + rate_coarse + count + rate_est).properties(width=600)
(rate_coarse + count + rate_est).properties(width=600)

alt.LayerChart(...)